In [1]:
DATA_DIR = "../../data"
MODELS_DIR = "../../models"

In [2]:
import pandas as pd

df = pd.read_csv(f"{DATA_DIR}/raw/results.csv")
df["date"] = pd.to_datetime(df["date"])

display(df)

,Unnamed: 0,date,team_1,team_2,_map,result_1,result_2,map_winner,starting_ct,ct_1,t_2,t_1,ct_2,event_id,match_id,rank_1,rank_2,map_wins_1,map_wins_2,match_winner
0,0,2020-03-18,Recon 5,TeamOne,Dust2,0,16,2,2,0,1,0,15,5151,2340454,62,63,0,2,2
1,1,2020-03-18,Recon 5,TeamOne,Inferno,13,16,2,2,8,6,5,10,5151,2340454,62,63,0,2,2
2,2,2020-03-18,New England Whalers,Station7,Inferno,12,16,2,1,9,6,3,10,5243,2340461,140,118,12,16,2
3,3,2020-03-18,Rugratz,Bad News Bears,Inferno,7,16,2,2,0,8,7,8,5151,2340453,61,38,0,2,2
4,4,2020-03-18,Rugratz,Bad News Bears,Vertigo,8,16,2,2,4,5,4,11,5151,2340453,61,38,0,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45768,45768,2015-11-05,G2,E-frag.net,Inferno,13,16,2,1,8,7,5,9,1970,2299059,7,16,1,2,2
45769,45769,2015-11-05,G2,E-frag.net,Dust2,16,13,1,1,10,5,6,8,1970,2299059,7,16,1,2,2
45770,45770,2015-11-04,CLG,Liquid,Inferno,16,12,1,1,7,8,9,4,1934,2299011,10,14,16,12,1
45771,45771,2015-11-03,NiP,Dignitas,Train,16,4,1,2,4,1,12,3,1934,2299001,6,12,16,4,1


In [3]:
df_test = pd.read_csv(f"{DATA_DIR}/preprocessed/test.csv")

display(df_test)

,elo_diff,winrate_10_diff,winrate_30_diff,experience_diff,rank_diff,h2h_winrate,team_1_wins
0,1.139435,0.231677,0.815696,-0.003897,-0.967555,-0.076338,1
1,-0.850737,-1.735120,-2.123590,-0.155839,3.686040,-0.076338,0
2,0.040321,0.161434,-0.154268,-0.231810,-0.578346,-0.076338,1
3,0.945895,1.074590,-1.006661,2.989363,-0.256825,-0.076338,1
4,-1.066911,-0.892207,0.420992,-2.475489,3.110686,-0.076338,0
...,...,...,...,...,...,...,...
4567,-0.613810,0.793619,-0.653947,-0.201422,0.538517,-1.544189,0
4568,-0.763782,0.793619,-0.653947,-0.201422,0.538517,-1.544189,0
4569,-0.367522,0.512648,-0.623276,-0.003897,0.521595,1.391514,0
4570,0.093768,0.512648,0.404196,-2.597043,0.132385,-0.076338,0


In [4]:
from keras.models import load_model

model = load_model(f"{MODELS_DIR}/model.keras")

display(model.summary())  # type: ignore

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │         1,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 16)             │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,157 (20.15 KB)

 Trainable params: 1,665 (6.50 KB)

 Non-trainable params: 160 (640.00 B)

 Optimizer params: 3,332 (13.02 KB)

None

In [5]:
import joblib

preprocessor = joblib.load(f"{MODELS_DIR}/preprocessor.joblib")

In [6]:
# Performance on the test dataset
from sklearn.metrics import accuracy_score, brier_score_loss, roc_auc_score

target = "team_1_wins"

X_test = df_test.drop(target, axis=1)
y_test = df_test[target]

y_proba = model.predict(X_test).ravel()  # type: ignore
y_pred = (y_proba >= 0.5).astype(int)

df_test_predicted = pd.concat([df_test, pd.Series(y_pred, name="prediction")], axis=1)

display(df_test_predicted)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, y_proba))
print("Brier score:", brier_score_loss(y_test, y_proba))

143/143 ━━━━━━━━━━━━━━━━━━━━ 0s 475us/step


,elo_diff,winrate_10_diff,winrate_30_diff,experience_diff,rank_diff,h2h_winrate,team_1_wins,prediction
0,1.139435,0.231677,0.815696,-0.003897,-0.967555,-0.076338,1,1
1,-0.850737,-1.735120,-2.123590,-0.155839,3.686040,-0.076338,0,0
2,0.040321,0.161434,-0.154268,-0.231810,-0.578346,-0.076338,1,1
3,0.945895,1.074590,-1.006661,2.989363,-0.256825,-0.076338,1,1
4,-1.066911,-0.892207,0.420992,-2.475489,3.110686,-0.076338,0,0
...,...,...,...,...,...,...,...,...
4567,-0.613810,0.793619,-0.653947,-0.201422,0.538517,-1.544189,0,0
4568,-0.763782,0.793619,-0.653947,-0.201422,0.538517,-1.544189,0,0
4569,-0.367522,0.512648,-0.623276,-0.003897,0.521595,1.391514,0,1
4570,0.093768,0.512648,0.404196,-2.597043,0.132385,-0.076338,0,1


Accuracy: 0.760061242344707
AUC: 0.8428204657845083
Brier score: 0.16121238470077515


In [7]:
import sys

sys.path.append("../../src/shared")

from data_prep import match_to_features  # type: ignore

# Do a test prediction on a made up match
future_match = {
    "team_1": "Rugratz",
    "team_2": "Bad News Bears",
    "rank_1": 61,
    "rank_2": 38,
    "date": pd.Timestamp("2024-06-01"),
}

X = match_to_features(preprocessor, df, future_match)
p = model.predict(X)[0][0]  # type: ignore

print(f"{future_match['team_1']} win probability: {p:.2%}")
print(
    "Predicted winner:", future_match["team_1"] if p > 0.5 else future_match["team_2"]
)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
Rugratz win probability: 18.38%
Predicted winner: Bad News Bears
